# 1. How can we distinguish between public and privately owned tree canopy in Boston using geospatial data and land use classifications?


**Dataset:** [Tree Canopy Metrics Dataset](https://bostonopendata-boston.opendata.arcgis.com/datasets/boston::canopy-change-assessment-tree-canopy-metrics/about?layer=4)

## Formulas and Definitions for Calculations

[Tree Ordinance 2023](https://www.boston.gov/sites/default/files/file/2024/01/City%20of%20Boston%20Chapter%2014%20of%20the%20Ordinances%20of%202023%20-%20Ordinance%20Establishing%20Protections%20for%20the%20City%20of%20Boston%20Tree%20Canopy%20-%20City%20Clerk%20signature.pdf)

**City of Boston Owned Parcels:** is any parcel of land that is owned by the City of Boston or quasi-agencies. Includes,
* City of Boston
* Boston Water and Sewer Commission
* Boston Housing Authority
* Boston Public Schools
* Boston Redevelopment Authority
* Boston Planning and Development Agency

We included any OWNER that had "City of Boston" + "_", example "Fire Department"

**Non-City of Boston Owned Parcels:** is any parcel of land that is not owned by the City of Boston. Includes state or federally owned parcels. 

**Percentage of Tree Canopy of City of Boston Owned Parcels**  $$\frac{\text{Total Tree canopy existing area classifed as City of Boston Owned}}{\text{Total Tree canopy existing area}}$$


**Percentage of Tree Canopy of Non-City of Boston Owned Parcels** $$\frac{\text{Total Tree canopy existing area classifed as Non-City Owned}}{\text{Total Tree canopy existing area}}$$

**TC_E_A:** Tree canopy existing area. The area of tree canopy present when viewed from above using aerial or satellite imagery, excluding water.

In [43]:
pip install thefuzz

Note: you may need to restart the kernel to use updated packages.


In [44]:
import pandas as pd
import numpy as np
from thefuzz import fuzz

# read Parcels Tree Canopy Metric Dataset layer from Analyze Boston 
# Data reflects 2019 tree canopy area aggregated at the parcel level
tree_canopy_metrics = pd.read_csv('datasets/PARCELS_Tree_Canopy_Metrics.csv', dtype={
    'PID': 'string'} )

# load citywide land audit dataset -- considered the authoritative source on what is city owned
# reflects 2022 ownership
city_land_audit_public = pd.read_csv("datasets/city_land_audit_public.csv", dtype={
    'Parcel': 'string'})

/var/folders/3k/5zd1hcfd4jv0bjvll1h2r7kr0000gn/T/ipykernel_8356/428630726.py:7: DtypeWarning: Columns (6,12,14,18,27) have mixed types. Specify dtype option on import or set low_memory=False.
  tree_canopy_metrics = pd.read_csv('datasets/PARCELS_Tree_Canopy_Metrics.csv', dtype={


# Exploring the Data

In [45]:
tree_canopy_metrics.head()

,FID,FID_Boston,BOSTON_LAN,FID_COB_FY,PID,CM_ID,GIS_ID,OWNER,ST_NUM,ST_NAME,...,TC_Pv_A,TC_Land_A,TC_Pi_A,TC_P_A,TC_E_P,TC_Pv_P,TC_P_P,TC_Pi_P,Shape__Area,Shape__Length
0,1,1,1,-1,,,,,,,...,5717593,218739022,38429015,44146608,18.670309,2.613888,20.182319,17.568431,2.244137e+08,8.914611e+06
1,2,1,1,1,0100001000,,0100001000,PASCUCCI CARLO,104 A 104,PUTNAM,...,0,1237,13,13,0.404204,0.000000,1.050930,1.050930,1.233845e+03,1.594376e+02
2,3,1,1,2,0100002000,,0100002000,ATANASOV DANIEL,197,LEXINGTON,...,0,1162,242,242,0.000000,0.000000,20.826162,20.826162,1.161436e+03,1.566642e+02
3,4,1,1,3,0100003000,,0100003000,CHEVARRIA ANA S,199,LEXINGTON,...,1,1144,211,212,0.699301,0.087413,18.531469,18.444056,1.145411e+03,1.558082e+02
4,5,1,1,4,0100004000,,0100004000,"MADDALENI JAMES E, TS",201,LEXINGTON,...,106,1191,102,208,0.587741,8.900084,17.464316,8.564232,1.192610e+03,1.571514e+02


In [46]:
tree_canopy_metrics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173875 entries, 0 to 173874
Data columns (total 72 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   FID            173875 non-null  int64  
 1   FID_Boston     173875 non-null  int64  
 2   BOSTON_LAN     173875 non-null  int64  
 3   FID_COB_FY     173875 non-null  int64  
 4   PID            173875 non-null  string 
 5   CM_ID          173875 non-null  object 
 6   GIS_ID         173875 non-null  object 
 7   OWNER          173875 non-null  object 
 8   ST_NUM         173875 non-null  object 
 9   ST_NAME        173875 non-null  object 
 10  ST_NAME_SU     173875 non-null  object 
 11  UNIT_NUM       173875 non-null  object 
 12  ZIPCODE        173875 non-null  object 
 13  LU             173875 non-null  object 
 14  PTYPE          173875 non-null  object 
 15  LOTSIZE        173875 non-null  int64  
 16  MAIL_ADDRE     173875 non-null  object 
 17  MAIL_CS        173875 non-nul

In [47]:
tree_canopy_metrics.columns

Index(['FID', 'FID_Boston', 'BOSTON_LAN', 'FID_COB_FY', 'PID', 'CM_ID',
       'GIS_ID', 'OWNER', 'ST_NUM', 'ST_NAME', 'ST_NAME_SU', 'UNIT_NUM',
       'ZIPCODE', 'LU', 'PTYPE', 'LOTSIZE', 'MAIL_ADDRE', 'MAIL_CS',
       'MAIL_ZIP', 'OWN_OCC', 'GROSS_AREA', 'BLDG_AREA', 'AV_TOTAL', 'AV_LAND',
       'AV_BLDG', 'GROSS_TAX', 'NUM_FLOORS', 'PID_LONG', 'FULL_ADDRE', 'CITY',
       'STArea__', 'STLength__', 'Shape_STAr', 'Shape_STLe', 'LU_GENERAL',
       'TC_ID', 'Shape_Leng', 'OBJECTID', 'TC_ID_1', 'Total_A', 'Can_A',
       'Grass_A', 'Soil_A', 'Water_A', 'Build_A', 'Road_A', 'Paved_A',
       'Perv_A', 'Imperv_A', 'Can_P', 'Grass_P', 'Soil_P', 'Water_P',
       'Build_P', 'Road_P', 'Paved_P', 'Perv_P', 'Imperv_P', 'OBJECTID_1',
       'TC_ID_12', 'VALUE_0', 'TC_E_A', 'TC_Pv_A', 'TC_Land_A', 'TC_Pi_A',
       'TC_P_A', 'TC_E_P', 'TC_Pv_P', 'TC_P_P', 'TC_Pi_P', 'Shape__Area',
       'Shape__Length'],
      dtype='object')

In [48]:
# count total number of unique owners 
len(tree_canopy_metrics["OWNER"].unique())

134787

In [49]:
# Count most common unique owners in the dataset
owner_count = tree_canopy_metrics['OWNER'].value_counts().to_frame(name='Count').reset_index()

# Standardize the names of the owners to be all uppercase
owner_count['OWNER'] = owner_count['OWNER'].str.upper()

# Show top 20 most common owners in Boston
owner_count.head(20)

,OWNER,Count
0,CITY OF BOSTON,2007
1,CITY OF BOSTON BY FCL,415
2,COMMONWEALTH OF MASS,226
3,TRUSTEES OF BOSTON COLLEGE,203
4,BOSTON HOUSING AUTHORITY,201
5,MARKETPLACE LOFTS LIMITED,162
6,COMMWLTH OF MASS,144
7,RELATED LOVEJOY RESIDENTIAL,133
8,BUCKMINSTER HOTEL CORP,132
9,BOSTON REDEVELOPMENT AUTH,132


## Data Cleaning

The dataset groups all ROW data together into one row, but left ownership as a blank space " " instead of null. Thus, we had to alter the dataset to encode this as ROW data because we do not know if it is entirely public or private. 

* Classified roads, sidewalks and streets as ROW data by setting the owner == "ROW"

In [50]:
# check the value of the parcel ID's to ensure they are all 10 digit codes 
tree_canopy_metrics['pid_length'] = tree_canopy_metrics['PID'].apply(lambda x: len(str(x)) if pd.notnull(x) else 0)

print(tree_canopy_metrics['pid_length'].value_counts())

pid_length
10    173874
1          1
Name: count, dtype: int64


In [51]:
# Replace any rows with a ' '  owner with a null
tree_canopy_metrics["OWNER"] = tree_canopy_metrics["OWNER"].replace(" ", pd.NA)

# find all null rows
tree_canopy_metrics[tree_canopy_metrics["OWNER"].isna()]

# Classify the null row owner as 'ROW', since it represents all roads and sidewalks in Boston
# Streets can be public or private depending on ownership, but the Parcels_tree_canopy_metric data doesn't have unique records for each street
tree_canopy_metrics.loc[tree_canopy_metrics["OWNER"].isna(), "OWNER"] = "ROW"

# Supplementary Question:
## 1. What percentage of Boston's tree canopy is located on public land versus residential parcels?

### Analysis 

In [52]:
# List of known owners of public parcels of land
public_entities = ["CITY OF BOSTON",
                   "BOSTON WATER AND SEWER COMMISSION",
                   "BOSTON HOUSING AUTHORITY",
                   "BOSTON PUBLIC SCHOOLS",
                   "BOSTON REDEVELOPMENT AUTHORITY",
                   "BOSTON PLANNING AND DEVELOPMENT AGENCY"
]

# Stores Fuzz matching results
scores = []

# Loop through each owner and calculate best token_set_ratio
for owner in owner_count['OWNER']:
    best_score = max([fuzz.token_set_ratio(owner, term) for term in public_entities])
    scores.append({
        "OWNER": owner,
        "SCORE": best_score
    })

fuzz_unfiltered_owner_output = pd.DataFrame(scores)

In [53]:
# Export to a csv to check for any misclassifications
fuzz_unfiltered_owner_output[fuzz_unfiltered_owner_output["SCORE"] > 78].to_csv('classifications/fuzz_unfiltered_owner_output.csv')

In [54]:
#after manually flagging errors reload the data
fuzz_classified_owners = pd.read_csv("classifications/fuzz_classified_owner_output.csv")

#fill in na's with 0's
fuzz_classified_owners = fuzz_classified_owners.fillna({'Flagged': 0})

#drop any non-public owners that were flagged by a 1 in the error column 
fuzz_public_owners = fuzz_classified_owners[fuzz_classified_owners["Flagged"] == 0]["OWNER"].tolist()

#returns list of public owners 
fuzz_public_owners, len(fuzz_public_owners)

(['CITY OF BOSTON',
  'CITY OF BOSTON BY FCL',
  'BOSTON HOUSING AUTHORITY',
  'BOSTON REDEVELOPMENT AUTH',
  'BOSTON HOUSING AUTH',
  'CITY OF BOSTON FCL',
  'BOSTON REDEVELOPMNT AUTH',
  'BOSTON REDVLPMNT AUTH',
  'BOSTON REDEVELOPMENT',
  'BOSTON REDEVLPMNT AUTH',
  'CITY OF BOSTON MUNICIPAL CP',
  'BOSTON REDEVLPMNT AUTHOR',
  'BOSTON WATER & SEWER',
  'BOSTON REDVLPMNT AUTHOR',
  'BOSTON REDEVELPMENT AUTH',
  'BOSTON REDEVELPMNT AUTH',
  'BOSTON WATER & SEWER COMM',
  'BOSTON REDEVELOPMENT LLC',
  'BOSTON DEVELOPMENT',
  'CITY OF BOSTON (POLICE)',
  'CITY OF BOSTON PARKS AND',
  'CITY OF BOSTON PWD',
  'CITY OF BOSTON SCHOOL DEPT',
  'CITY  OF  BOSTON',
  'CITY OF BOSTON PUBLIC HEALTH',
  'BOSTON REDEVELOPMENT AUTHOR',
  'CITY OF BOSTON DND',
  'BOSTON REDEVELOP AUTHORITY',
  'BOSTON REDEVLOPMENT AUTH',
  'CITY OF BOSTON (POLICE DEPT)',
  'CITY OF BOSTON FCL.',
  'CITY OF BOSTON - DND',
  'CITY OF BOSTON CREDIT UNION',
  'CITY OF BOSTON PARKS',
  'CITY OF BOSTON TRST',
  'BOSTON P

In [55]:
# relabel all parcels in the dataset to be either 'city owned' or 'non city owned' based on ownership
# any parcels with owners included in the fuzz_public_owners list were considered City of Boston owned, all else was considered Non-city owned
# ROW data was excluded for consistency since there are public and private roads/sidewalks/streets
tree_canopy_metrics["ownership"] = tree_canopy_metrics["OWNER"].apply(lambda x: 
                                "city owned" if x in fuzz_public_owners
                                 else "ROW" if x == "ROW" 
                                 else "non-city owned")

# Group TC_E_A(Tree canopy existing area) by public/private/row ownership
agg_tree_canopy = tree_canopy_metrics.groupby("ownership")["TC_E_A"].sum().reset_index()

# Add total tree canopy to the dataframe
agg_tree_canopy["Total Tree Canopy Area"] = tree_canopy_metrics["TC_E_A"].sum()

# Calculate Percentage of Tree Canopy
agg_tree_canopy["Percentage of Total Tree Canopy"] = ((agg_tree_canopy["TC_E_A"] / agg_tree_canopy["Total Tree Canopy Area"])*100).round(2)

# export this table to be a csv for visualizing 
agg_tree_canopy.to_csv("visualization_datasets/city_non_city_owned.csv")

agg_tree_canopy

,ownership,TC_E_A,Total Tree Canopy Area,Percentage of Total Tree Canopy
0,ROW,40839251,357959872,11.41
1,city owned,66113930,357959872,18.47
2,non-city owned,251006691,357959872,70.12


In [56]:
tree_canopy_metrics.loc[tree_canopy_metrics["PID"] == " ", "PID"] = "ROW"

In [57]:
#export classifications for visualizing q6
#the file has the PID and ownership columns which allows classification of a geospatial file in ArcGIS Pro
tree_canopy_metrics[["PID", "ownership"]].to_csv("ArcGIS_merge_classification_table.csv")

# 3. What are the trends in tree canopy loss or gain (2014-2019) for public and private areas?

Dataset: [Parcel Tree Canopy Change Metrics](https://data.boston.gov/dataset/canopy-change-assessment-tree-canopy-change-metrics)

In [58]:
# load in the tree canopy change dataset
parcel_canopy_change_metrics = pd.read_csv("datasets/PARCELS_Tree_Canopy_Change_Metrics.csv")

# the polygon representing all null data has a " " for the owner
# Replace the OWNER == " " as 'ROW', since it represents all roads and sidewalks in Boston
parcel_canopy_change_metrics.loc[parcel_canopy_change_metrics["OWNER"]== " ", "OWNER"] = "ROW"

# apply the classification method to the owners column
# column 'ownership' classifies parcels as 'city owned', 'non-city owned', or 'ROW'
parcel_canopy_change_metrics["ownership"] = parcel_canopy_change_metrics["OWNER"].apply(lambda x: 
                                "city owned" if x in fuzz_public_owners
                                 else "ROW" if x == "ROW" 
                                 else "non-city owned")


/var/folders/3k/5zd1hcfd4jv0bjvll1h2r7kr0000gn/T/ipykernel_8356/2653908250.py:2: DtypeWarning: Columns (4,6,12,14,18,27) have mixed types. Specify dtype option on import or set low_memory=False.
  parcel_canopy_change_metrics = pd.read_csv("datasets/PARCELS_Tree_Canopy_Change_Metrics.csv")


In [59]:
parcel_canopy_change_metrics

,FID,FID_Boston,BOSTON_LAN,FID_COB_FY,PID,CM_ID,GIS_ID,OWNER,ST_NUM,ST_NAME,...,TreeCanopy,TreeCano_1,Change_Are,Change_Per,TreeCano_2,TreeCano_3,Change_P_1,Shape__Area,Shape__Length,ownership
0,1,1,1,-1,,,,ROW,,,...,3.986780e+07,4.087817e+07,1.010378e+06,2.534321,18.240958,18.703243,0.462284,2.244137e+08,8.914611e+06,ROW
1,2,1,1,1,0100001000,,0100001000,PASCUCCI CARLO,104 A 104,PUTNAM,...,7.893076e-01,5.009830e+00,4.220522e+00,534.711924,0.063840,0.405203,0.341363,1.233845e+03,1.594376e+02,non-city owned
2,3,1,1,2,0100002000,,0100002000,ATANASOV DANIEL,197,LEXINGTON,...,2.350375e+00,0.000000e+00,-2.350375e+00,-100.000000,0.202640,0.000000,-0.202640,1.161436e+03,1.566642e+02,non-city owned
3,4,1,1,3,0100003000,,0100003000,CHEVARRIA ANA S,199,LEXINGTON,...,2.794576e+01,7.557041e+00,-2.038872e+01,-72.958186,2.440145,0.659860,-1.780286,1.145411e+03,1.558082e+02,non-city owned
4,5,1,1,4,0100004000,,0100004000,"MADDALENI JAMES E, TS",201,LEXINGTON,...,9.597758e-01,5.904437e+00,4.944661e+00,515.189183,0.080560,0.495599,0.415038,1.192610e+03,1.571514e+02,non-city owned
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173870,173871,1,1,17020,300452750,0300450000,300450000,WHITTIER PLACE CONDOMINIUM,6 -8,WHITTIER,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,2.214873e+05,1.990351e+03,non-city owned
173871,173872,1,1,17021,300452752,0300450000,300450000,WHITTIER PLACE CONDOMINIUM,6 -8,WHITTIER,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,2.214873e+05,1.990351e+03,non-city owned
173872,173873,1,1,17022,300452754,0300450000,300450000,WHITTIER PLACE CONDOMINIUM,6 -8,WHITTIER,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,2.214873e+05,1.990351e+03,non-city owned
173873,173874,1,1,17023,300452756,0300450000,300450000,WHITTIER PLACE CONDOMINIUM,6 -8,WHITTIER,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,2.214873e+05,1.990351e+03,non-city owned


### Column Definitions:
* TreeCanopy= 2014 total canopy area (baseline)
* TreeCano_1 = 2019 total canopy area
* Gain = Tree canopy area gain 
* Loss = Tree canopy area loss 

## Analysis

In [60]:
# Find the difference between 2019 total canopy area and 2014 total canopy area for each parcel 
parcel_canopy_change_metrics["diff"] = parcel_canopy_change_metrics['TreeCano_1'] - parcel_canopy_change_metrics['TreeCanopy'] 

# Find the difference between tree canopy gain and loss for each parcel
parcel_canopy_change_metrics["diff_g_l"] = parcel_canopy_change_metrics['Gain'] - parcel_canopy_change_metrics['Loss'] 

#print results 
parcel_canopy_change_metrics[["Gain", "Loss","No_Change", 'TreeCanopy', 'TreeCano_1', 'diff', "diff_g_l"]]

,Gain,Loss,No_Change,TreeCanopy,TreeCano_1,diff,diff_g_l
0,7.443209e+06,6.432831e+06,3.343496e+07,3.986780e+07,4.087817e+07,1.010378e+06,1.010378e+06
1,4.220522e+00,0.000000e+00,7.893076e-01,7.893076e-01,5.009830e+00,4.220522e+00,4.220522e+00
2,0.000000e+00,2.350375e+00,0.000000e+00,2.350375e+00,0.000000e+00,-2.350375e+00,-2.350375e+00
3,3.760469e+00,2.414919e+01,3.796573e+00,2.794576e+01,7.557041e+00,-2.038872e+01,-2.038872e+01
4,4.944661e+00,0.000000e+00,9.597758e-01,9.597758e-01,5.904437e+00,4.944661e+00,4.944661e+00
...,...,...,...,...,...,...,...
173870,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
173871,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
173872,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
173873,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [61]:
#aggregate tree canopy gain and loss by ownership
gain_loss_agg = parcel_canopy_change_metrics.groupby("ownership")[["Gain", "Loss",'TreeCanopy', 'TreeCano_1', "diff_g_l"]].sum().reset_index()

#rename the columns 
gain_loss_agg = gain_loss_agg.rename(columns={"TreeCanopy": "2014 Tree Canopy", "TreeCano_1": "2019 Tree Canopy"})

#calculate the percentage difference in tree canopy from 2014 to 2019
gain_loss_agg["Percent_Diff"] = ((gain_loss_agg["2019 Tree Canopy"] -  gain_loss_agg["2014 Tree Canopy"])/gain_loss_agg["2014 Tree Canopy"]) *100

# #export dataset for visualization
# gain_loss_agg.to_csv("visualization_datasets/gain_loss_agg.csv")
gain_loss_agg

,ownership,Gain,Loss,2014 Tree Canopy,2019 Tree Canopy,diff_g_l,Percent_Diff
0,ROW,7.443209e+06,6.432831e+06,3.986780e+07,4.087817e+07,1.010378e+06,2.534321
1,city owned,5.468686e+06,4.190453e+06,6.486015e+07,6.613838e+07,1.278234e+06,1.970753
2,non-city owned,2.716149e+07,2.920234e+07,2.531463e+08,2.511055e+08,-2.040853e+06,-0.806195


# 4. Which areas have the highest potential for expanding tree canopy, and are they public or private spaces?


## Column Definitions
* TC_PV_A= Possible vegetation area. Grass or shrub area that is theoretically available for
the establishment of tree canopy.
* TC_Pi_A= Possible impervious area. Asphalt or concrete surfaces or bare soil, excluding
roads and buildings, that are theoretically available for the establishment of tree canopy.
* TC_P_A= Possible area. Area theoretically available for establishment of tree canopy.
* TC_Land_A = Land area. Land area excluding water bodies.

In [62]:
potential_expansion = tree_canopy_metrics.groupby("ownership")[["TC_Pv_A", "TC_Pi_A",'TC_P_A', "TC_Land_A"]].sum().reset_index()

potential_expansion["Percentage Possible Vegetation Area"] = potential_expansion["TC_Pv_A"]/potential_expansion["TC_Land_A"]

potential_expansion["Percentage Possible Impervious Area"] = potential_expansion["TC_Pi_A"]/potential_expansion["TC_Land_A"]

potential_expansion["Percentage Possible Tree Canopy Area"] = round((potential_expansion["TC_P_A"]/potential_expansion["TC_Land_A"])*100)

potential_expansion[["ownership", "Percentage Possible Tree Canopy Area"]]

,ownership,Percentage Possible Tree Canopy Area
0,ROW,20.0
1,city owned,47.0
2,non-city owned,41.0


## Analysis

In [63]:
potential_expansion

,ownership,TC_Pv_A,TC_Pi_A,TC_P_A,TC_Land_A,Percentage Possible Vegetation Area,Percentage Possible Impervious Area,Percentage Possible Tree Canopy Area
0,ROW,5717593,38429015,44146608,218739022,0.026139,0.175684,20.0
1,city owned,49344353,25743512,75087865,158821456,0.310691,0.162091,47.0
2,non-city owned,191896429,204930391,396826820,967899129,0.198261,0.211727,41.0


# Additional Question: Who are the most common owners of "Non-City" owned parcels?

In [64]:
# filter parcels to non-city owned parcels
non_city_parcels = tree_canopy_metrics[tree_canopy_metrics['ownership'] == "non-city owned"]

# filter to parcels that are tax exempt
# have 'PTYPE' that start with 9
# Assume these parcels are publicly owned 
tax_exempt = non_city_parcels[non_city_parcels['PTYPE'].astype(str).str.startswith('9')]

# Find owners of who own the most parcels of land 
tax_exempt_owners = pd.DataFrame(tax_exempt["OWNER"].value_counts()).reset_index()

tax_exempt_owners.head(20)

,OWNER,count
0,COMMONWEALTH OF MASS,226
1,COMMWLTH OF MASS,144
2,MASS BAY TRANSPORTATION AUTH,130
3,MASS BAY TRANSP AUTH,97
4,MARRIOTT OWNERSHIP RESORTS,84
5,BOSTON UNIVERSITY TRSTS OF,81
6,DUDLEY NEIGHBORS INC,80
7,MASS PORT AUTHORITY,76
8,MASSACHUSETTS PORT AUTHORITY,76
9,NORTHEASTERN UNIVERSITY,57


In [65]:
# What land use codes are showing the largest volume of parcels?
# Find PTYPEs of who own the most parcels of land 
tax_exempt_ptype = pd.DataFrame(tax_exempt["PTYPE"].value_counts()).reset_index()

tax_exempt_ptype.head(10)

,PTYPE,count
0,995,9536
1,985,1469
2,986,619
3,907,364
4,977,363
5,905,360
6,985,220
7,906,214
8,970,197
9,979,183


* 995 = CONDO MAIN
* 985 = OTHER EXEMPT BLDG
* 986 = VACANT LAND
* 907 = EXEMPT 121A PROPERTY
* 977 = COLLEGE
* 905 = CHARITABLE ORGANIZATION
* 985 = OTHER EXEMPT BLDG
* 906 = RELIGIOUS ORGANIZATION
* 970 = CHURCH, SYNAGOGUE
* 979 = HOSPITAL


In [66]:
tax_exempt_lu = pd.DataFrame(tax_exempt["LU_GENERAL"].value_counts()).reset_index()

tax_exempt_lu.head(10)

,LU_GENERAL,count
0,Residential,9919
1,Institutional,4720


In [67]:
# Create a pivot table to count parcels by OWNER and LU_GENERAL category
pivot_table = pd.pivot_table(
    tax_exempt,
    index="OWNER",
    columns="LU_GENERAL",
    values="PID",
    aggfunc="count",
    fill_value=0  
)

# Optional: sort by total parcels across categories
pivot_table["TOTAL"] = pivot_table.sum(axis=1)
pivot_table = pivot_table.sort_values(by="TOTAL", ascending=False)

pivot_table.head(10)

LU_GENERAL,Institutional,Residential,TOTAL
OWNER,,,
COMMONWEALTH OF MASS,220,6,226
COMMWLTH OF MASS,144,0,144
MASS BAY TRANSPORTATION AUTH,130,0,130
MASS BAY TRANSP AUTH,97,0,97
MARRIOTT OWNERSHIP RESORTS,84,0,84
BOSTON UNIVERSITY TRSTS OF,81,0,81
DUDLEY NEIGHBORS INC,0,80,80
MASSACHUSETTS PORT AUTHORITY,76,0,76
MASS PORT AUTHORITY,74,2,76


In [68]:
# filter to state owned entities
# most of these ptypes correlate to a property occupancy code
# equal to "Commonwealth of Massachusetts" in some abbrevation
# this is a list that can be added to
state_ptype = ["901","914","923","924","925","929","965"]

state_owned_parcels = tax_exempt[tax_exempt['PTYPE'].isin(state_ptype)][["PID","OWNER"]]
state_owned_parcels["OWNER"].value_counts()

OWNER
MASSACHUSETTS PORT AUTHORITY    12
MASS PORT AUTHORITY              4
MASSACHUSETTS PORT AUTH          3
COMMWLTH OF MASS                 2
MASSACHUSETT PORT AUTHORITY      1
COMMONWLTH OF MASS               1
Name: count, dtype: int64

# Citywide Land Audit Exploration 


In [70]:
city_land_audit_public['pid_length'] = city_land_audit_public['Parcel'].apply(lambda x: len(str(x)) if pd.notnull(x) else 0)

print(city_land_audit_public['pid_length'].value_counts())

pid_length
10    2939
9        1
Name: count, dtype: int64


In [71]:
# drop the row that has a 9 digit PID code
print(city_land_audit_public[city_land_audit_public['pid_length'] == 9])

      OID_     Parcel                               Address     Owner  \
2920  2921  401862000  320-350 BROOKLINE AV BOSTON MA 02115  Non-City   

     Care_and_Custody                    Prop_NAME  \
2920         Non-City  Ambulance 16 & Paramedic 16   

                                    GlobalID User_Agency  Lot_Size_SF  \
2920  {4F6B0E5E-C36C-4AB1-9641-EED3EE9FD26E}         EMS          NaN   

      Shape_Length  Shape_Area shape_wkt  pid_length  
2920           0.0         0.0       NaN           9  


In [72]:
# drop Parcel == 401862000 because it also does not show in the geospatial map
city_land_audit_public = city_land_audit_public[city_land_audit_public['Parcel'] != '401862000']
print(len(city_land_audit_public)) #check it was dropped. should return 2939

2939


In [73]:
#find all unique values of parcel ID's in the land audit and the tree canopy metrics datasets
land_audit_pid = city_land_audit_public["Parcel"].astype(str).str.strip().unique().tolist()
tree_metrics = tree_canopy_metrics["PID"].astype(str).str.strip().unique().tolist()

# find the number of parcel ID's that are in the land audit from 2022, but are not in the tree canopy metrics dataset
missing_parcels_from_tree_canopy_metrics = [item for item in land_audit_pid if item not in tree_metrics]

print("There are", len(missing_parcels_from_tree_canopy_metrics), "parcel ID's missing from the tree_canopy_metrics dataset that are considered public in the city wide land audit. This is most likely due to changed parcel mapping from 2019 to 2022.")

There are 33 parcel ID's missing from the tree_canopy_metrics dataset that are considered public in the city wide land audit. This is most likely due to changed parcel mapping from 2019 to 2022.


# EDA: Compare with Citywide Land Audit

In [74]:
# Is there any PID's considered owned by City of Boston in the land audit that are not considered public by using the fuzz
filtered_city_owned = tree_canopy_metrics[tree_canopy_metrics["ownership"] == 'city owned']

# pids considered city owned by the fuzzy matching
filtered_city_owned_pids = filtered_city_owned["PID"].astype(str).str.strip().unique().tolist()

# pids considered city owned by the land audit 
land_audit_pids = city_land_audit_public["Parcel"].astype(str).str.strip().unique().tolist()

In [75]:
# Find items in the land audit that were not found through Fuzzy matching 
audit_not_in_fuzz_match = [item for item in land_audit_pids if item not in filtered_city_owned_pids]
len(audit_not_in_fuzz_match )

223

In [76]:
# returns rows in tree canopy metrics that were considered public in the land audit, but missed by fuzzy matching 
# consider these rows city-owned by appending them to our list considered city-owned
tree_canopy_metrics[tree_canopy_metrics['PID'].isin(audit_not_in_fuzz_match)][["PID","OWNER"]]

,PID,OWNER
865,0100925000,PARKS & RECREATION
3822,0104382000,TRUSTEES OF PUBLIC LIBRARY
3892,0104437000,PUBLIC FACILITIES COMMISSION
5448,0106313100,EBCDC INC
6595,0200638000,COMMUNITY PROVIDERS OF
...,...,...
143591,1001487000,BACK OF THE HILL CONDO TRUST
147604,0203511400,STARBOARD LEASEHOLD
153608,0303822110,BRADLEY C STEPHEN MANAGER
155112,0203510175,BASILICA LEASEHOLD


In [77]:
tree_canopy_metrics[tree_canopy_metrics['PID'].isin(audit_not_in_fuzz_match)]["OWNER"].value_counts()

OWNER
ECONOMIC DEVELOPMENT AND        27
BACK OF THE HILL CONDO TRUST     6
MASSACHUSETTS PORT               4
BOSTON CENTER FOR THE ARTS       3
WELD ST ASSOCS LP                3
                                ..
DEPT OF ENVIRONMENTAL            1
SMOOKLER DEBORAH C               1
CROSSTOWN CENTER COMMON AREA     1
CROSSTOWN CENTER HOTEL LLC       1
METROPOLITAN PRIMARY CONDO       1
Name: count, Length: 143, dtype: int64

In [78]:
# check why a parcel would be included in the audit, but not fuzzy matching 
query = tree_canopy_metrics[tree_canopy_metrics['PID'].isin(audit_not_in_fuzz_match)]
query[query["OWNER"] == "MASSACHUSETTS PORT"]

,FID,FID_Boston,BOSTON_LAN,FID_COB_FY,PID,CM_ID,GIS_ID,OWNER,ST_NUM,ST_NAME,...,TC_Pi_A,TC_P_A,TC_E_P,TC_Pv_P,TC_P_P,TC_Pi_P,Shape__Area,Shape__Length,pid_length,ownership
16343,16344,1,1,58972,0602674205,,602674205,MASSACHUSETTS PORT,20,FID KENNEDY,...,1339563,1351770,1.387613,0.826999,91.579611,90.752612,1.482477e+06,9743.478392,10,non-city owned
89711,89712,1,1,58972,0602674205,,602674205,MASSACHUSETTS PORT,20,FID KENNEDY,...,0,0,0.000000,0.000000,0.000000,0.000000,3.168091e+01,1412.600464,10,non-city owned
89712,89713,1,1,58972,0602674205,,602674205,MASSACHUSETTS PORT,20,FID KENNEDY,...,0,0,0.000000,0.000000,0.000000,0.000000,1.538330e+00,174.779366,10,non-city owned
89714,89715,1,1,58972,0602674205,,602674205,MASSACHUSETTS PORT,20,FID KENNEDY,...,0,0,0.000000,0.000000,0.000000,0.000000,9.026611e+00,140.239663,10,non-city owned


In [79]:
# the parcel above in tree canopy metrics was owned by Mass Port in 2018, which we did not consider city-owned
# the parcel in 2022 seems to be under the care of the BPDA
city_land_audit_public[city_land_audit_public["Parcel"] == "0602674205"]

,OID_,Parcel,Address,Owner,Care_and_Custody,Prop_NAME,GlobalID,User_Agency,Lot_Size_SF,Shape_Length,Shape_Area,shape_wkt,pid_length
2600,2601,0602674205,"20 FID KENNEDY DR, South Boston Waterfront: R...",BPDA,Tenant,NaN,{BCF0A821-B47A-4C6F-B1AD-DF78907BD922},Non City,1395715.0,0.037787,0.000016,NaN,10


In [80]:
# Find items in the fuzzy matching that were not considered city owned in the land audit
fuzz_not_in_land_audit = [item for item in filtered_city_owned_pids if item not in land_audit_pids]
len(fuzz_not_in_land_audit)

360

In [81]:
# returns owner names and counts of parcels that were not considered city owned by the audit, but were found by fuzzy matching to be city owned
additional_fuzz_matches = tree_canopy_metrics[tree_canopy_metrics['PID'].isin(fuzz_not_in_land_audit)][["PID","OWNER"]]
additional_fuzz_matches["OWNER"].value_counts()

OWNER
CITY OF BOSTON                  145
BOSTON HOUSING AUTHORITY         80
CITY OF BOSTON BY FCL            74
BOSTON HOUSING AUTH              12
BOSTON REDEVELOPMENT             10
BOSTON REDEVELOPMENT AUTH        10
BOSTON REDEVELOPMENT LLC          8
BOSTON DEVELOPMENT                7
CITY  OF  BOSTON                  3
BOSTON REDEVELPMENT AUTH          3
BOSTON REDVLPMNT AUTH             2
BOSTON REDEVELOPMNT AUTH          1
BOSTON REDEVELOPMENT AUTHOR       1
CITY OF BOSTON - DND              1
CITY OF BOSTON PWD                1
CITY OF BOSTON FCL                1
BOSTON REDVLPMNT AUTHOR           1
BOSTON REDEVLPMNT AUTH            1
BOSTON REDEVELOPMENT AUTHORI      1
CITY OF BOSTON FIRE DEPARTME      1
BOSTON PUBLIC HEALTH              1
Name: count, dtype: int64